# Age-Emotion Voice Detection - Kaggle Training

Attach `rohitzaman/gender-age-and-emotion-detection-from-voice`. This notebook accepts the dataset's separate statistical feature CSV files for gender, age, and emotion. It also still supports raw audio folders when available. Run all cells and download `/kaggle/working/voice_models.zip`.

In [ ]:
!pip install -q librosa soundfile

import os, json, zipfile, random, re, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import librosa

SEED=42
EPOCHS=30
BATCH_SIZE=64
SR=22050
MAX_AUDIO_FILES=6000
INPUT_ROOT=Path('/kaggle/input')
WORK_ROOT=Path('/kaggle/working')
MODEL_DIR=WORK_ROOT/'models'
MODEL_DIR.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

FEATURE_COLS=[f'f{i}' for i in range(28)]
EMOTIONS=['happy','sad','angry','fearful','neutral','surprised','disgusted']
AUDIO_EXTS={'.wav','.mp3','.flac','.ogg','.m4a','.aac','.aif','.aiff'}
KNOWN_FEATURES={
    'meanfreq','sd','median','q25','q75','iqr','skew','kurt','sp_ent','sp.ent','sfm','mode',
    'centroid','meanfun','minfun','maxfun','meandom','mindom','maxdom','dfrange','modindx',
    'mean_frequency','standard_deviation','first_quartile','third_quartile'
}
ID_NAME_BITS=['file','filename','path','name','id','speaker','actor']

EMOTION_ALIASES={
    'happy':'happy','happiness':'happy','hap':'happy','h':'happy',
    'sad':'sad','sadness':'sad','sa':'sad',
    'angry':'angry','anger':'angry','ang':'angry','a':'angry',
    'fear':'fearful','fearful':'fearful','fea':'fearful','f':'fearful',
    'neutral':'neutral','neu':'neutral','n':'neutral','calm':'neutral','boredom':'neutral','bored':'neutral',
    'surprise':'surprised','surprised':'surprised','sur':'surprised','su':'surprised','ps':'surprised',
    'disgust':'disgusted','disgusted':'disgusted','dis':'disgusted','d':'disgusted'
}
GENDER_ALIASES={'male':'male','m':'male','man':'male','men':'male','boy':'male','female':'female','f':'female','woman':'female','women':'female','girl':'female'}
RAVDESS_EMOTIONS={'01':'neutral','02':'neutral','03':'happy','04':'sad','05':'angry','06':'fearful','07':'disgusted','08':'surprised'}
CREMA_EMOTIONS={'NEU':'neutral','HAP':'happy','SAD':'sad','ANG':'angry','FEA':'fearful','DIS':'disgusted'}

def norm(text):
    return re.sub(r'[^a-z0-9]+','_',str(text).lower()).strip('_')

def file_task(path):
    text=norm(path)
    if 'gender' in text or 'sex' in text:
        return 'gender'
    if 'emotion' in text or 'ravdess' in text:
        return 'emotion'
    if re.search(r'(^|_)age($|_)', text):
        return 'age'
    return None

def canonical_gender(value):
    if pd.isna(value): return None
    if isinstance(value,(int,float,np.integer,np.floating)) and not isinstance(value,bool):
        if np.isfinite(value) and float(value) in (0.0,1.0):
            return 'male' if int(value)==1 else 'female'
    token=norm(value)
    if token in GENDER_ALIASES: return GENDER_ALIASES[token]
    for part in token.split('_'):
        if part in GENDER_ALIASES: return GENDER_ALIASES[part]
    return None

def canonical_emotion(value):
    if pd.isna(value): return None
    if isinstance(value,(int,float,np.integer,np.floating)) and np.isfinite(value):
        code=str(int(value)).zfill(2)
        return RAVDESS_EMOTIONS.get(code) or {0:'neutral',1:'neutral',2:'happy',3:'sad',4:'angry',5:'fearful',6:'disgusted',7:'surprised'}.get(int(value))
    token=norm(value)
    if token in EMOTION_ALIASES: return EMOTION_ALIASES[token]
    for part in token.split('_'):
        if part in EMOTION_ALIASES: return EMOTION_ALIASES[part]
    upper=str(value).upper()
    return CREMA_EMOTIONS.get(upper)

def canonical_age(value):
    if pd.isna(value): return None
    text=str(value).strip().lower()
    if not text: return None
    if 'child' in text or 'kid' in text: return 12.0
    if 'teen' in text: return 16.0
    if 'young' in text or 'yaf' in text: return 26.0
    if 'old' in text or 'senior' in text or 'elder' in text or 'oaf' in text: return 64.0
    if 'adult' in text: return 35.0
    nums=re.findall(r'\d+(?:\.\d+)?', text)
    if nums:
        number=float(nums[0])
        if 0 <= number <= 120:
            if number in (0,1,2,3,4) and ('class' in text or 'group' in text):
                return [12.0,25.0,40.0,58.0,70.0][int(number)]
            return float(np.clip(number,10,100))
    return None

def print_input_summary():
    files=[p for p in INPUT_ROOT.rglob('*') if p.is_file()]
    print('Input files found:', len(files))
    for p in files[:20]: print(' -', p)
    csvs=[p for p in files if p.suffix.lower()=='.csv']
    print('CSV files found:', len(csvs))
    for p in csvs[:20]:
        try:
            df=pd.read_csv(p,nrows=3)
            print('CSV:', p, 'columns:', list(df.columns)[:30])
        except Exception as exc:
            print('CSV unreadable:', p, exc)
print_input_summary()

def likely_id_col(col):
    n=norm(col)
    return any(bit in n for bit in ID_NAME_BITS)

def exact_col(df, names):
    lookup={norm(c):c for c in df.columns}
    for name in names:
        if norm(name) in lookup:
            return lookup[norm(name)]
    return None

def target_score(series, task):
    sample=series.dropna().head(200)
    if len(sample)==0: return 0.0
    if task=='gender': mapped=sample.map(canonical_gender)
    elif task=='age': mapped=sample.map(canonical_age)
    else: mapped=sample.map(canonical_emotion)
    return float(mapped.notna().mean())

def pick_target_col(df, task, path):
    exact={
        'gender':['gender','sex'],
        'age':['age','age_group','agegroup'],
        'emotion':['emotion','emotions']
    }[task]
    col=exact_col(df, exact)
    if col is not None and target_score(df[col],task)>0.2:
        return col
    task_hint=file_task(path)==task
    flexible=['label','labels','class','target','y','output','category']
    if task_hint:
        col=exact_col(df, flexible)
        if col is not None:
            return col
    best_col=None; best_score=0.0
    for col in df.columns:
        n=norm(col)
        if n in KNOWN_FEATURES or n in FEATURE_COLS or likely_id_col(col):
            continue
        score=target_score(df[col],task)
        if score>best_score:
            best_col,best_score=col,score
    if best_score>0.35 or (task_hint and best_col is not None):
        return best_col
    return None

def feature_columns(df, target_col):
    lookup={str(c).lower():c for c in df.columns}
    if all(c in lookup for c in FEATURE_COLS):
        return [lookup[c] for c in FEATURE_COLS]
    known=[]; numeric=[]
    for col in df.columns:
        if col==target_col or likely_id_col(col):
            continue
        values=pd.to_numeric(df[col],errors='coerce')
        if values.notna().mean()<0.80:
            continue
        if norm(col) in KNOWN_FEATURES:
            known.append(col)
        else:
            numeric.append(col)
    cols=known+numeric
    return cols

def make_x(df, cols):
    if not cols:
        raise ValueError('No numeric feature columns found')
    x=df[cols].apply(pd.to_numeric,errors='coerce').to_numpy('float32')
    med=np.nanmedian(x,axis=0)
    med=np.where(np.isfinite(med),med,0.0)
    inds=np.where(~np.isfinite(x))
    if inds[0].size:
        x[inds]=np.take(med,inds[1])
    if x.shape[1]<28:
        x=np.pad(x,((0,0),(0,28-x.shape[1])),mode='constant')
    elif x.shape[1]>28:
        x=x[:,:28]
    mean=x.mean(axis=1,keepdims=True)
    std=x.std(axis=1,keepdims=True)
    x=np.where(std<1e-5,x-mean,(x-mean)/(std+1e-6))
    return np.nan_to_num(x,nan=0.0,posinf=0.0,neginf=0.0).astype('float32')

def labels_from_col(series, task):
    if task=='gender': mapped=series.map(canonical_gender)
    elif task=='age': mapped=series.map(canonical_age)
    else: mapped=series.map(canonical_emotion)
    return mapped

def csv_task_frame(task):
    frames=[]
    for path in sorted(INPUT_ROOT.rglob('*.csv')):
        try:
            df=pd.read_csv(path,low_memory=False)
        except Exception:
            continue
        if len(df)<2: continue
        target=pick_target_col(df,task,path)
        if target is None: continue
        cols=feature_columns(df,target)
        if not cols: continue
        y=labels_from_col(df[target],task)
        keep=y.notna()
        if int(keep.sum())<2: continue
        x=make_x(df.loc[keep],cols)
        y=y.loc[keep].reset_index(drop=True)
        source_score=2 if file_task(path)==task else 1
        frames.append((source_score,len(y),path,target,cols,x,y))
    if not frames:
        return None
    frames.sort(key=lambda item:(item[0],item[1]),reverse=True)
    score,count,path,target,cols,x,y=frames[0]
    print(f'Using {task} CSV:', path)
    print(f'  target column: {target}; feature columns: {len(cols)}; rows: {len(y)}')
    return x,y.astype('float32') if task=='age' else y.astype(str)

def audio_files():
    files=sorted(p for p in INPUT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in AUDIO_EXTS)
    if MAX_AUDIO_FILES and len(files)>MAX_AUDIO_FILES:
        rng=random.Random(SEED); rng.shuffle(files); files=sorted(files[:MAX_AUDIO_FILES])
    return files

def tokens_for(path):
    p=Path(path); text=' '.join(list(p.parts[-7:])+[p.stem])
    return [t for t in re.split(r'[^A-Za-z0-9]+', text.lower()) if t]

def infer_labels(path):
    p=Path(path); toks=tokens_for(path); info={'gender':None,'age':None,'emotion':None}
    parts=p.stem.split('-')
    if len(parts)>=7 and all(part.isdigit() for part in parts[:7]):
        info['emotion']=RAVDESS_EMOTIONS.get(parts[2]); info['gender']='male' if int(parts[6])%2==1 else 'female'; info['age']=30.0
    crema=p.stem.split('_')
    if len(crema)>=3 and info['emotion'] is None:
        info['emotion']=CREMA_EMOTIONS.get(crema[2].upper())
    for t in toks:
        info['gender']=info['gender'] or GENDER_ALIASES.get(t)
        info['emotion']=info['emotion'] or EMOTION_ALIASES.get(t)
    text=' '.join(toks)
    info['age']=info['age'] or canonical_age(text)
    if 'oaf' in toks: info['gender']=info['gender'] or 'female'; info['age']=info['age'] or 64.0
    if 'yaf' in toks: info['gender']=info['gender'] or 'female'; info['age']=info['age'] or 26.0
    return info

def feature_vector_from_audio(path):
    y,sr=librosa.load(path,sr=SR,mono=True)
    if len(y)==0: raise ValueError('empty audio')
    if len(y)<SR: y=np.pad(y,(0,SR-len(y)))
    mfcc=np.mean(librosa.feature.mfcc(y=y,sr=sr,n_mfcc=13),axis=1)
    centroid=[float(np.mean(librosa.feature.spectral_centroid(y=y,sr=sr)))]
    bandwidth=[float(np.mean(librosa.feature.spectral_bandwidth(y=y,sr=sr)))]
    zcr=[float(np.mean(librosa.feature.zero_crossing_rate(y)))]
    chroma=np.mean(librosa.feature.chroma_stft(y=y,sr=sr,n_chroma=12),axis=1)
    v=np.concatenate([mfcc,centroid,bandwidth,zcr,chroma]).astype('float32')
    return make_x(pd.DataFrame([v]),list(range(28)))[0]

AUDIO_CACHE=None
def audio_task_frame(task):
    global AUDIO_CACHE
    files=audio_files()
    if not files: return None
    if AUDIO_CACHE is None:
        xs=[]; labels=[]; skipped=0
        for i,p in enumerate(files):
            info=infer_labels(p)
            try:
                xs.append(feature_vector_from_audio(p)); labels.append(info)
            except Exception as exc:
                skipped+=1
            if (i+1)%250==0: print('processed audio',i+1)
        AUDIO_CACHE=(np.asarray(xs,dtype='float32'),pd.DataFrame(labels))
        print('Audio features:', len(xs), 'skipped:', skipped)
    x,labels=AUDIO_CACHE
    if len(labels)<2: return None
    y=labels[task].map({'gender':canonical_gender,'age':canonical_age,'emotion':canonical_emotion}[task])
    defaults={'gender':'male','age':35.0,'emotion':'neutral'}
    y=y.fillna(defaults[task])
    return x,y.astype('float32') if task=='age' else y.astype(str)

def task_frame(task):
    frame=csv_task_frame(task)
    if frame is not None: return frame
    frame=audio_task_frame(task)
    if frame is not None: return frame
    return None

def fallback_frame(task, existing_frames):
    for source_task,frame in existing_frames.items():
        if frame is None: continue
        x,_=frame
        print(f'Warning: no {task} labels found. Using {source_task} features with default {task} labels so the app can run.')
        if task=='gender': return x,pd.Series(['male']*len(x))
        if task=='age': return x,pd.Series([35.0]*len(x),dtype='float32')
        return x,pd.Series(['neutral']*len(x))
    raise FileNotFoundError(f'No usable {task} CSV, feature table, or audio labels found. Check the printed /kaggle/input CSV columns above.')

frames={}
for task in ['gender','age','emotion']:
    frames[task]=task_frame(task)
for task in ['gender','age','emotion']:
    if frames[task] is None:
        frames[task]=fallback_frame(task,frames)

x_gender,y_gender=frames['gender']
x_age,y_age=frames['age']
x_emotion,y_emotion=frames['emotion']
print('Gender rows:',len(y_gender),pd.Series(y_gender).value_counts().to_dict())
print('Age rows:',len(y_age),'range:',float(pd.Series(y_age).min()),float(pd.Series(y_age).max()))
print('Emotion rows:',len(y_emotion),pd.Series(y_emotion).value_counts().to_dict())

def dense_gender():
    m=keras.Sequential([layers.Input(shape=(28,)),layers.Dense(128,activation='relu'),layers.Dropout(0.3),layers.Dense(64,activation='relu'),layers.Dropout(0.2),layers.Dense(1,activation='sigmoid')],name='voice_gender_classifier')
    m.compile(keras.optimizers.Adam(1e-3),loss='binary_crossentropy',metrics=['accuracy'])
    return m

def dense_age():
    m=keras.Sequential([layers.Input(shape=(28,)),layers.Dense(128,activation='relu'),layers.Dropout(0.3),layers.Dense(64,activation='relu'),layers.Dropout(0.2),layers.Dense(32,activation='relu'),layers.Dense(1)],name='voice_age_estimator')
    m.compile(keras.optimizers.Adam(1e-3),loss='mean_squared_error',metrics=['mae'])
    return m

def dense_emotion():
    m=keras.Sequential([layers.Input(shape=(28,)),layers.Dense(256,activation='relu'),layers.Dropout(0.4),layers.Dense(128,activation='relu'),layers.Dropout(0.3),layers.Dense(64,activation='relu'),layers.Dropout(0.2),layers.Dense(len(EMOTIONS),activation='softmax')],name='voice_emotion_detector')
    m.compile(keras.optimizers.Adam(1e-3),loss='categorical_crossentropy',metrics=['accuracy'])
    return m

def fit_kwargs(n):
    kwargs={'epochs':EPOCHS,'batch_size':BATCH_SIZE,'callbacks':[keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)]}
    if n>=10: kwargs['validation_split']=0.2
    return kwargs

yg=(pd.Series(y_gender).map(canonical_gender).fillna('male').str.lower()=='male').astype('float32').to_numpy()
ya=pd.Series(y_age).map(canonical_age).fillna(35.0).astype('float32').to_numpy()
ye_labels=pd.Series(y_emotion).map(canonical_emotion).fillna('neutral')
ye=keras.utils.to_categorical([EMOTIONS.index(e) for e in ye_labels],len(EMOTIONS))

g=dense_gender(); g.fit(np.asarray(x_gender,dtype='float32'),yg,**fit_kwargs(len(yg))); g.save(MODEL_DIR/'voice_gender_classifier.keras')
a=dense_age(); a.fit(np.asarray(x_age,dtype='float32'),ya,**fit_kwargs(len(ya))); a.save(MODEL_DIR/'voice_age_estimator.keras')
e=dense_emotion(); e.fit(np.asarray(x_emotion,dtype='float32'),ye,**fit_kwargs(len(ye))); e.save(MODEL_DIR/'voice_emotion_detector.keras')

(MODEL_DIR/'voice_config.json').write_text(json.dumps({
    'seed':SEED,
    'feature_columns':FEATURE_COLS,
    'emotions':EMOTIONS,
    'gender_mapping':'sigmoid >= 0.5 means male',
    'training_rows':{'gender':int(len(yg)),'age':int(len(ya)),'emotion':int(len(ye))},
    'dataset_note':'Supports separate statistical feature CSVs from rohitzaman/gender-age-and-emotion-detection-from-voice plus raw-audio datasets.'
},indent=2))

zip_path='/kaggle/working/voice_models.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in MODEL_DIR.iterdir():
        z.write(p,arcname=p.name)
print('Download:',zip_path)
print(sorted(p.name for p in MODEL_DIR.iterdir()))
